# 04 - Simulation Truth And Lateral Distance

Use GCD geometry and simulation truth to compute the perpendicular distance between pulsed in-ice DOMs and a truth-track candidate.


In [ ]:
from pathlib import Path
import sys
sys.path.append(str(Path.cwd().parent / 'src'))

import matplotlib.pyplot as plt
import pandas as pd

from icetray_tutorial.paths import SIMULATION
from icetray_tutorial.frames import event_header_dict, iter_frames, stop_name
from icetray_tutorial.geometry import inice_dom_positions, particle_like_keys, pulsed_dom_lateral_distances, read_geometry
from icetray_tutorial.pulses import summarize_pulses


In [ ]:
geometry = read_geometry(SIMULATION.gcd)
positions = inice_dom_positions(geometry)
len(positions)


In [ ]:
xy = pd.DataFrame([{'string': omkey.string, 'om': omkey.om, 'x': pos[0], 'y': pos[1], 'z': pos[2]} for omkey, pos in positions.items()])
ax = xy.plot.scatter(x='x', y='y', s=4, alpha=0.4, figsize=(6, 6))
ax.set_aspect('equal')
ax.set_title('InIce DOM positions')


In [ ]:
MIN_HIT_DOMS = 1
MAX_PHYSICS_FRAMES = 500

distance_columns = [
    'run_id', 'event_id', 'sub_event_id', 'sub_event_stream',
    'string', 'om', 'truth_key', 'distance_m', 'event_hit_doms',
]
rows = []
diagnostics = {
    'physics_frames_seen': 0,
    'frames_with_pulses': 0,
    'frames_passing_dom_threshold': 0,
    'frames_with_truth_distances': 0,
}
first_pulse_key = None
first_particle_like_truth_keys = []
first_interesting_keys = []

for frame in iter_frames(SIMULATION.event_file):
    if stop_name(frame) not in ('Physics', 'P'):
        continue

    diagnostics['physics_frames_seen'] += 1
    if not first_interesting_keys:
        first_interesting_keys = [
            key for key in frame.keys()
            if any(word in key.lower() for word in ('pulse', 'mc', 'muon', 'primary', 'truth'))
        ]
        first_particle_like_truth_keys = particle_like_keys(frame)

    pulse_summary = summarize_pulses(frame)
    if pulse_summary is None:
        if diagnostics['physics_frames_seen'] >= MAX_PHYSICS_FRAMES:
            break
        continue

    diagnostics['frames_with_pulses'] += 1
    first_pulse_key = first_pulse_key or pulse_summary.pulse_key
    if pulse_summary.hit_doms < MIN_HIT_DOMS:
        if diagnostics['physics_frames_seen'] >= MAX_PHYSICS_FRAMES:
            break
        continue

    diagnostics['frames_passing_dom_threshold'] += 1
    distances = pulsed_dom_lateral_distances(frame, geometry)
    if distances:
        diagnostics['frames_with_truth_distances'] += 1
        header = event_header_dict(frame)
        for row in distances:
            rows.append({**header, **row, 'event_hit_doms': pulse_summary.hit_doms})

    if diagnostics['physics_frames_seen'] >= MAX_PHYSICS_FRAMES:
        break

distance_table = pd.DataFrame(rows, columns=distance_columns)
diagnostics, first_pulse_key, first_particle_like_truth_keys[:20], first_interesting_keys[:40], distance_table.head()


In [ ]:
if distance_table.empty:
    print('No lateral-distance rows were built.')
    print('Diagnostics:', diagnostics)
    print('First pulse key found:', first_pulse_key)
    print('Particle-like truth keys found:', first_particle_like_truth_keys)
    print('Interesting keys from first Physics frame:', first_interesting_keys)
    print('Try setting a truth key explicitly in pulsed_dom_lateral_distances(..., truth_key=...) after choosing from the available keys.')
else:
    distance_table['distance_m'].dropna().plot.hist(bins=60)
    plt.xlabel('DOM lateral distance to truth track [m]')
    plt.ylabel('pulsed DOMs')


## Practice

1. Inspect keys containing `MC`, `Muon`, or `Primary`.
2. Raise `MIN_HIT_DOMS` after the first successful plot.
3. Choose a better truth key if the frame provides one.
